In [ ]:
# Accuracy report (label-free): any CPU runtime, ~2-5 min. Reads every PS4 results folder in MyDrive/mytronD.
from google.colab import drive
drive.mount("/content/drive")
import zipfile, glob, os, shutil, subprocess, sys
D = "/content/drive/MyDrive/mytronD"
zs = [z for z in glob.glob(f"{D}/*.zip") if any(n.endswith("tools/accuracy_report.py") for n in zipfile.ZipFile(z).namelist())]
assert zs, "upload the newest ps4v4.zip (with tools/accuracy_report.py) to " + D
z = max(zs, key=os.path.getmtime); print("code from", z)
shutil.rmtree("/content/_acc", ignore_errors=True); zipfile.ZipFile(z).extractall("/content/_acc")
script = min(glob.glob("/content/_acc/**/tools/accuracy_report.py", recursive=True), key=len)
# every folder whose sub-folders hold hands.jsonl = a results folder (ps4v5_FINAL, second account's folder, ...)
results = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{D}/*/*/hands.jsonl")})
slam = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{D}/*/*/world_stats.json")})
print("results folders:", results, "\nSLAM folders:", slam)
cmd = [sys.executable, script, *results, "--out", f"{D}/accuracy_report"] + (["--slam", *slam] if slam else [])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
import pandas as pd
pd.read_csv(f"{D}/accuracy_report/per_clip.csv")
